In [42]:
from datasets import load_dataset
import pandas as pd
from collections import defaultdict
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import json
import re
from torch.utils.data import TensorDataset, DataLoader
import kenlm

import random
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import os
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter

# Bước chuẩn bị

### Load dataset train/test/valid

In [2]:
# Load dataset
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 56445
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
    test: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
})


In [3]:
# Chia tập train/test/valid
# Note: df_valid dùng để thử các kết quả model trước khi áp dụng lần cuối với test
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])
df_valid = pd.DataFrame(dataset['validation'])

In [ ]:
df_train

,input,target
0,zinredie zidne: htlv có lương befo nhkấtl lịc ...,zinedine zidane: hlv có lương bèo nhất lịch sử...
1,Còn cứ kéo dài như vậy sẽ ko ổn.,Còn cứ kéo dài như vậy sẽ không ổn.
2,PVC trien khai nhieu cong trinh xay dung lon.,PVC triển khai nhiều công trình xây dựng lớn.
3,"""bữa cơm thwuwowrg khnôg có gì cao nơưn mỹ vị,...","""bữa cơm thưởng không có gì cao lương mỹ vị, c..."
4,"Dieu tra vu vo hui hang ti dong o Nam Can, Ca ...","Điều tra vụ vỡ hụi hàng tỉ đồng ở Năm Căn, Cà ..."
...,...,...
56440,May mà được ông làm tóc có tâm. Đến tết mái có...,May mà được ông làm tóc có tâm. Đến tết mái có...
56441,Nguoi dep Philippines dang quang HH The gioi 2...,Người đẹp Philippines đăng quang HH Thế giới 2...
56442,smartfone - người bn đồng hànk.,smartphone - người bạn đồng hành.
56443,"trongv vụ xử edan, cknúg ta cos lúg túng....","trong vụ xử vedan, chúng ta có lúng túng...."


### load and create vocabulary

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Tạo 1 vocab dựa trên 1 dataset khác
vocab = []
with open(r"/content/vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx và ngược lại
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [4]:
# Tạo 1 vocab dựa trên 1 dataset khác
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx và ngược lại
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [ ]:
print(len(vocab))
print(vocab[:10])

77814
['a', 'abagiua', 'abatoa', 'a_bàng', 'a_bung', 'a_di', 'a_di_đà_kinh', 'adiđà_phật', 'a_di_đà_phật', 'a_di_đà_tam_tôn']


Tạo 1 Dictionary là counts để tính số lần xuất hiện của mỗi từ trong vocab\
Lợi ích: sau này ta sẽ cộng điểm ưu tiên cho thằng có số lần xuất hiện lớn hơn\

In [5]:
counts = defaultdict(int)

for sentence in df_train['target']:
    sentence = sentence.lower()
    words = sentence.split()

    for i in range(len(words)):
        word = words[i]
        counts[word] += 1

Load danh sách các stopwords

In [5]:
with open(r"/content/vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

In [6]:
with open(r"D:\NLP\project\vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

# Các thuật toán sửa lỗi sẽ sử dụng:
(Note: thuật toán sửa lỗi 1, 2 là ở trong Symspell Correction, mình tách ra code riêng chứ k xài hàm có sẵn)

### Thuật toán 1: Sửa lỗi chính tả của từ
Sử dụng Symmetric Delete:
Chỉ sử dụng phép Xóa (Delete) để thu hẹp không gian tìm kiếm.\
Nếu hai từ có khoảng cách chỉnh sửa (Edit Distance) là $k$, chúng sẽ có chung ít nhất một biến thể "xóa đi $n$ ký tự" ($n \le k$).

Import file từ 1 chữ tiếng Việt -> nguyên âm - phụ âm - dấu câu trong telex\
VD: 'ắ' - 'a', 'w', 's'

In [7]:
with open(r"D:\NLP\project\telex.txt", "r", encoding="utf-8") as f:
    telex = f.read()

telex = re.sub(r',\s*}', '\n}', telex)
telex = json.loads(telex)
telex

{'ă': ['a', 'w', ''],
 'â': ['a', 'a', ''],
 'ê': ['e', 'e', ''],
 'ô': ['o', 'o', ''],
 'ơ': ['o', 'w', ''],
 'ư': ['u', 'w', ''],
 'đ': ['d', 'd', ''],
 'á': ['a', '', 's'],
 'à': ['a', '', 'f'],
 'ã': ['a', '', 'x'],
 'ả': ['a', '', 'r'],
 'ạ': ['a', '', 'j'],
 'ắ': ['a', 'w', 's'],
 'ằ': ['a', 'w', 'f'],
 'ẵ': ['a', 'w', 'x'],
 'ẳ': ['a', 'w', 'r'],
 'ặ': ['a', 'w', 'j'],
 'ấ': ['a', 'a', 's'],
 'ầ': ['a', 'a', 'f'],
 'ẫ': ['a', 'a', 'x'],
 'ẩ': ['a', 'a', 'r'],
 'ậ': ['a', 'a', 'j'],
 'é': ['e', '', 's'],
 'è': ['e', '', 'f'],
 'ẽ': ['e', '', 'x'],
 'ẻ': ['e', '', 'r'],
 'ẹ': ['e', '', 'j'],
 'ế': ['e', 'e', 's'],
 'ề': ['e', 'e', 'f'],
 'ễ': ['e', 'e', 'x'],
 'ể': ['e', 'e', 'r'],
 'ệ': ['e', 'e', 'j'],
 'í': ['i', '', 's'],
 'ì': ['i', '', 'f'],
 'ĩ': ['i', '', 'x'],
 'ỉ': ['i', '', 'r'],
 'ị': ['i', '', 'j'],
 'ó': ['o', '', 's'],
 'ò': ['o', '', 'f'],
 'õ': ['o', '', 'x'],
 'ỏ': ['o', '', 'r'],
 'ọ': ['o', '', 'j'],
 'ố': ['o', 'o', 's'],
 'ồ': ['o', 'o', 'f'],
 'ỗ': ['o', 'o'

Thử nghiệm hàm

In [8]:
def create_telex_form(word):
    word = word.lower()
    prefix = ""      # Phụ âm đầu
    vowel_base = ""  # Nguyên âm gốc
    suffix = ""      # Phụ âm cuối
    word_tone = ""   # Dấu thanh
    word_mod = ""    # Ký tự gõ mũ/móc

    VOWELS = "aeiouy" # Các nguyên âm tiếng Việt
    state = 0 # 0: phụ âm đầu, 1: nguyên âm

    i = 0
    while i < len(word):
        step = 1
        # Trường hợp 'ươ'
        if i < len(word) - 1 and word[i:i+2] in telex:
            char = word[i:i+2]
            step = 2
        else:
            char = word[i]

        # Nếu ký tự nằm trong từ điển
        if char in telex:
            if char == 'đ':
                if state == 0: prefix += 'dd'
                else: suffix += 'dd'
            else:
                vowel_base += telex[char][0]
                if telex[char][1]: word_mod = telex[char][1]
                if telex[char][2]: word_tone = telex[char][2]
                state = 1

        # Nếu ký tự là chữ bình thường
        else:
            if char in VOWELS:
                vowel_base += char
                state = 1
            else:
                if state == 0: prefix += char
                else: suffix += char

        i += step

    # Tạo biến thể
    variants = set()
    inline_vowel = vowel_base + word_mod

    # Kiểu 1 & 2: Gõ chuẩn ngay sau nguyên âm hoặc ném dấu thanh ra cuối
    variants.add(prefix + inline_vowel + word_tone + suffix)
    variants.add(prefix + inline_vowel + suffix + word_tone)

    # Kiểu 3: Phím bổ nghĩa (w, a, e, o) ném ra cuối từ
    if word_mod:
        variants.add(prefix + vowel_base + word_tone + suffix + word_mod)
        variants.add(prefix + vowel_base + suffix + word_mod + word_tone)
        variants.add(prefix + vowel_base + suffix + word_tone + word_mod)

    # Trường hợp đặc biệt của "ươ"
    if vowel_base == 'uo' and word_mod == 'w':
        # Tách w hai lần ngay sau nguyên âm
        variants.add(prefix + 'uwow' + word_tone + suffix)
        variants.add(prefix + 'uwow' + suffix + word_tone)

        # Tách w đầu, w cuối
        variants.add(prefix + 'uwo' + word_tone + suffix + 'w')
        variants.add(prefix + 'uwo' + suffix + 'w' + word_tone)
        variants.add(prefix + 'uwo' + suffix + word_tone + 'w')

    return list(v for v in variants if v)

In [ ]:
create_telex_form("trường")

['truwongwf',
 'truwowngf',
 'truwofngw',
 'truwongfw',
 'truofngw',
 'truongwf',
 'truongfw',
 'truwowfng',
 'truowfng',
 'truowngf']

Hàm tạo các biến thể xóa từ 0 đến k kí tự của từ, hàm trả về list các biến thể xóa

In [9]:
def get_deletes(word, k = 2):
    queue = [word]
    variant_list = set()

    for _ in range(k):
        temp_queue = set()
        for w in queue:
            if len(w) > 1:
                for i in range(len(w)):
                    variant = w[:i] + w[i+1:]
                    variant_list.add(variant)
                    temp_queue.add(variant)
        queue = temp_queue
    return variant_list

In [ ]:
# Kiểm tra hàm
print(get_deletes("chào"))

{'cào', 'ch', 'hà', 'co', 'cà', 'ho', 'ào', 'chà', 'cho', 'hào'}


In [10]:
# Tạo danh sách các từ thiếu từ 0 đến k kí tự trong vocab
# Tạo 1 Dictionary (Hash Map) để lưu toàn bộ biến thể xóa
sym_dict = defaultdict(list)

for word in vocab:
    # Lấy các biến thể và từ gốc của 1 từ
    base_forms = [word] + create_telex_form(word)

    for form in base_forms:
        # Lưu form này (distance 0)
        if word not in sym_dict[form]:
            sym_dict[form].append(word)

        # Tạo deletes cho từng form và lưu vào từ điển
        variant_list = get_deletes(form)
        for variant in variant_list:
            # Map biến thể tới từ gốc 'word' hiện tại nếu chưa map
            if word not in sym_dict[variant]:
                sym_dict[variant].append(word)

In [ ]:
# Kiểm tra
sym_dict

defaultdict(list,
            {'a': ['a',
              'aga',
              'ala',
              'ami',
              'avi',
              'à',
              'ả',
              'á',
              'ạ',
              'abc',
              'ác',
              'adn',
              'ag',
              'ai',
              'ải',
              'ái',
              'ak',
              'al',
              'am',
              'ám',
              'an',
              'án',
              'ang',
              'anh',
              'ao',
              'ào',
              'ảo',
              'áo',
              'áp',
              'ar',
              'as',
              'át',
              'atk',
              'au',
              'áy',
              'ăă',
              'ă',
              'ăn',
              'â',
              'âm',
              'ân',
              'âu',
              'ba',
              'bà',
              'bả',
              'bã',
              'bá',
              'bạ',
              '

In [11]:
# Hàm tính khoảng cách của xâu 1 và xâu 2 bằng thuật toán Damerau-Levenshtein
# Là hàm edit_distance nhưng có thêm phép đổi chỗ các kí tự (gõ lộn thứ tự)
# Cải thiện thêm bằng khoảng cách bàn phím cho trường hợp gõ nhầm

# Các phím liền kề trên bàn phím để tính trọng số
ADJACENT_KEYS = {
    'q': 'wea', 'w': 'qeasd', 'e': 'wrsdf', 'r': 'etdfg', 't': 'ryfgh', 'y': 'tughj', 'u': 'yihjk', 'i': 'uojkl', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwsz', 's': 'weadzx', 'd': 'ersfxc', 'f': 'rtdgcv', 'g': 'tyfhvb', 'h': 'yugjbn', 'j': 'uihknm', 'k': 'iojlm', 'l': 'opk',
    'z': 'asx', 'x': 'sdzc', 'c': 'dfxv', 'v': 'fgcb', 'b': 'ghvn', 'n': 'hjbm', 'm': 'jkn'
}

# Các phím mang tính năng của Telex
TELEX_KEYS = set(['s', 'f', 'r', 'x', 'j', 'w', 'a', 'e', 'o', 'd'])

def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]

    # Khởi tạo giá trị hàng và cột đầu tiên
    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            char1 = s1[i - 1]
            char2 = s2[j - 1]
            # Tính chi phí thay thế (0 nếu giống nhau, 0.5 nếu khác mà gần nhau trên bàn phím, 1 nếu khác và ở xa trên bàn phím)
            if char1 == char2:
                sub_cost = 0.0
            elif char1 in ADJACENT_KEYS.get(char2, "") or char2 in ADJACENT_KEYS.get(char1, ""):
                # Nếu gõ nhầm 2 phím cạnh nhau (VD: a và s), chi phí chỉ là 0.5
                sub_cost = 0.5
            else:
                # Lỗi gõ nhầm phím xa nhau, chi phí 1.0
                sub_cost = 1.0

            # Tính chi phí thêm / xóa
            del_cost = 0.8 if char1 in TELEX_KEYS else 1.0
            ins_cost = 0.8 if char2 in TELEX_KEYS else 1.0

            dp[i][j] = min(
                dp[i - 1][j] + del_cost,       # Xóa
                dp[i][j - 1] + ins_cost,       # Thêm
                dp[i - 1][j - 1] + sub_cost    # Thay thế
            )

            # Phép Đổi chỗ (Transposition)
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 0.5)

    return dp[n][m]

In [ ]:
# Kiểm tra hàm
edit_distance("truwowfng", "trwuwowfng")

0.8

In [121]:
# Hàm tìm từ gần nhất với từ nhập vào, hàm trả về các từ gần nhất và khoảng cách của nó
def lookup(word, k=2):
    # Trả về tuple (từ, khoảng cách) để đồng nhất với kết quả ở dưới
    # if word in vocab:
    #     return word

    variant_list = get_deletes(word)
    variant_list.add(word)

    # Lưu đáp án là các từ gần nhất và khoảng cách của nó
    candidates = {}

    # Tra cứu các biến thể
    for variant in variant_list:
        if variant in sym_dict:
            for suggestion in sym_dict[variant]:
                if suggestion in candidates:
                    continue

                # SỬ DỤNG HÀM TELEX ĐỂ TÍNH KHOẢNG CÁCH (NẾU BẠN KIỂM TRA LỖI GÕ PHÍM)
                telex_form = [suggestion] + create_telex_form(suggestion)

                dist = min([edit_distance(word, form) for form in telex_form])

                # Nhận khi khoảng cách <= k, tồn tại trong từ điển và có tần suất > 0
                if dist <= k and suggestion in vocab and counts.get(suggestion, 0) > 0:
                    candidates[suggestion] = (dist, counts.get(suggestion, 0))

    # Xếp hạng ưu tiên: Distance nhỏ trước -> Tần suất cao trước
    result = sorted(candidates.items(),
                    key=lambda x: (x[1][0], -x[1][1]))

    ans = []
    # Giải nén thẳng tuple cho dễ đọc, đổi tên biến tránh ghi đè tham số 'word'
    for cand_word, (dist, count) in result:
        # Sử dụng tham số k thay vì hardcode số 2
        if dist >= k:
            break

        # Trả về cả từ và khoảng cách
        ans.append(cand_word)

    return ans

In [ ]:
# Kiểm tra hàm
lookup("trwuwowfng")

['trường', 'trưởng', 'trương', 'tường', 'rường']

### Thuật toán 2: viết tắt (Teencode & Informal Variants)
Mapping (tạo các cặp viết tắt và từ chính) + Contextual Ranking (xem thử cái nào khả thi hơn)\
Kết hợp Deep Learning để xem xét ngữ cảnh của từ viết tắt từ đó tăng độ chính xác

### Thuật toán 3: Sửa lỗi viết không dấu
2 cách:
- 1 là N-gram + Viterbi
- 2 là model BARTpho-syllable nhỏ

### Phát hiện lỗi:
Bằng cách tra từ không thuộc từ điển và N-gram (tri-gram) của từ quá thấp thì từ đó sẽ lỗi

In [62]:
model_lm = kenlm.Model(r"D:/NLP/project/3-gram-lm.binary")

def detect_error_word(sentence):
    # model.full_scores trả về (prob, ngram_length, is_oov) cho từng từ
    # trừ 1 là do ở cuối nó thêm </s>
    scores = list(model_lm.full_scores(sentence))[:-1]

    error_indices = []
    words = sentence.split()
    for i, (prob, length, oov) in enumerate(scores):
        if prob < -4.4 or words[i] not in vocab:
            # print(words[i], prob)
            error_indices.append(i)

    return error_indices

In [ ]:
sentence = "nhiều giãi pháp đã áp dụng thành coong trong thực tiễn, có doanh thu tương dối lớn, phục vụ không chỏ cho doanh nghiệp mà còn cho đời sóng xã hội."
print(detect_error_word(sentence))
words = sentence.split()
for i in detect_error_word(sentence):
    print(words[i])

[1, 7, 15, 20, 28]
giãi
coong
dối
chỏ
sóng


### Model chung: Chọn từ phù hợp với ngữ cảnh
Dựa trên Skip-gram (window_size = 5) + log-Probability\
Hướng sử dụng là check tìm trong câu các từ không có trong từ điển -> gán lỗi cho nó -> sử dụng thuật toán sửa lỗi 1 tìm ra top n từ có thể sẽ đúng (ở đây n = 5) -> xem thử có trong từ điển không -> xem từ nào gần nhất với các từ trong câu\
VD: hôm nay tôi đi hõc ở trường
- hõc không có trong từ điển -> sai -> các từ thay thế: học, hóc, húc, hạc, hắc
- tính điểm dựa trên các yếu tố sau và chọn thằng có điểm cao nhất
    - 1. Điểm dựa vào ngữ cảnh (skip-gram trừ stopwords)
    - 2. Điểm dựa vào tần xuất (n-gram)
    - 3. Điểm cộng nếu tạo được thành 1 từ có trong từ điển
    - 4. Điểm trừ nếu ứng cử viên là 1 từ trong stopword

Tạo các cặp skipgram với window_size = 5

In [13]:
window_size = 5
skipgram_data = []

for sentence in df_train['target']:
    # Lấy danh sách từ
    sentence = sentence.lower()
    words = sentence.split()

    # Lọc stopword
    filtered_words = []
    for word in words:
        if word not in word_to_idx or word in stopword:
            continue

        filtered_words.append(word)

    for i, word in enumerate(filtered_words):
        # Kiểm tra xem từ này có trong vocab
        if word not in word_to_idx:
            continue

        center = word_to_idx[word]

        for j in range(-window_size, window_size + 1):
            # Kiểm tra các khoảng cách xem có phù hợp (trừ chính nó)
            if j == 0:
                continue
            if 0 <= i + j < len(filtered_words):
                context_word = words[i + j]
                #Kiểm tra các từ bên cạnh có trong vocab không
                if context_word in word_to_idx:
                    context = word_to_idx[context_word]
                    skipgram_data.append((center, context))

In [13]:
len(skipgram_data)

1996340

model Skipgram

In [14]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        embed = self.embedding(x)
        out = self.linear(embed)

        return out

In [15]:
#  Chuẩn bị chạy bằng GPU
device = torch.device("cuda")

centers = torch.LongTensor([pair[0] for pair in skipgram_data])
contexts = torch.LongTensor([pair[1] for pair in skipgram_data])

batch_size = 8192
dataset = TensorDataset(centers, contexts)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

EMBED_DIM = 300
VOCAB_SIZE = len(vocab)
model_skipgram = SkipGram(VOCAB_SIZE, EMBED_DIM).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_skipgram.parameters(), lr=0.001)

In [11]:
epochs = 30

model_skipgram.train()

for epoch in range(epochs):
    total_loss = 0

    for batch_centers, batch_contexts in loader:

        batch_centers = batch_centers.to(device)
        batch_contexts = batch_contexts.to(device)

        # Forward
        outputs = model_skipgram(batch_centers)
        loss = criterion(outputs, batch_contexts)

        # Backward
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 2070.0903
Epoch 2, Loss: 1634.1326
Epoch 3, Loss: 1604.1827
Epoch 4, Loss: 1589.4813
Epoch 5, Loss: 1579.7079
Epoch 6, Loss: 1572.5143
Epoch 7, Loss: 1566.7907
Epoch 8, Loss: 1562.0750
Epoch 9, Loss: 1557.9741
Epoch 10, Loss: 1554.4234
Epoch 11, Loss: 1551.2863
Epoch 12, Loss: 1548.4097
Epoch 13, Loss: 1545.7901
Epoch 14, Loss: 1543.4017
Epoch 15, Loss: 1541.1702
Epoch 16, Loss: 1539.1074
Epoch 17, Loss: 1537.1604
Epoch 18, Loss: 1535.3403
Epoch 19, Loss: 1533.5965
Epoch 20, Loss: 1531.9565
Epoch 21, Loss: 1530.4060
Epoch 22, Loss: 1528.9458
Epoch 23, Loss: 1527.5280
Epoch 24, Loss: 1526.2009
Epoch 25, Loss: 1524.8747
Epoch 26, Loss: 1523.6405
Epoch 27, Loss: 1522.4366
Epoch 28, Loss: 1521.3057
Epoch 29, Loss: 1520.1528
Epoch 30, Loss: 1519.1027


Lưu model train skip-gram

In [12]:
torch.save(model_skipgram.state_dict(), 'model_skipgram.pth')
from google.colab import files

files.download('model_skipgram.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
model_path = r"D:/NLP/project/model_skipgram.pth"
model_skipgram.load_state_dict(torch.load(model_path, weights_only=True))

<All keys matched successfully>

Hàm tính độ liên quan của 2 từ

In [17]:
def calculate_similarity(word1, word2, embedding_matrix, word_to_idx):
    # Lấy vector của từng từ
    idx1 = word_to_idx[word1]
    idx2 = word_to_idx[word2]
    vec1 = embedding_matrix[idx1]
    vec2 = embedding_matrix[idx2]

    # Tính Cosine Similarity
    # Công thức: (A . B) / (||A|| * ||B||)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

Test thử với các từ "hóc xương", "học xương". "hóc xương" sẽ phải cho ra điểm cao hơn so với "học xương"

In [18]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

score1 = calculate_similarity("hóc", "xương", embedding_matrix, word_to_idx)
score2 = calculate_similarity("học", "xương", embedding_matrix, word_to_idx)

print(score1, score2)

0.023185026 -0.039693136


Test thử với các từ "sóng thần", "sóng tiền". "sóng thần" sẽ phải cho ra điểm cao hơn so với "sóng tiền"

In [19]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

score1 = calculate_similarity("sóng", "thần", embedding_matrix, word_to_idx)
score2 = calculate_similarity("sóng", "tiền", embedding_matrix, word_to_idx)

print(score1, score2)

0.046962634 -0.032947786


Hàm chọn từ sửa sai dựa trên độ liên quan

In [127]:
def choose_best_candidate(candidates, sentence, error_idx, embedding_matrix, error_indices, 
                          word_to_idx, stopwords, kenlm_model, window_size=3):
    best_word = None
    max_total_score = -float('inf')

    # 1. Lấy ngữ cảnh cho Word2Vec (giữ nguyên hoặc thu hẹp lại một chút)
    w2v_window = 2 # Thu hẹp lại để tránh lấy các từ quá xa không liên quan
    start_w2v = max(0, error_idx - w2v_window)
    end_w2v = min(len(sentence), error_idx + w2v_window + 1)
    context_words = [sentence[i] for i in range(start_w2v, end_w2v) 
                     if i not in error_indices and sentence[i] not in stopwords]

    for candidate in candidates:
        # --- PHẦN 1: ĐIỂM NGỮ NGHĨA (Word2Vec) ---
        avg_sim = 0.0
        if candidate in word_to_idx:
            sim_sum = 0
            valid_contexts = 0
            for context in context_words:
                if context in word_to_idx:
                    sim = calculate_similarity(candidate, context, embedding_matrix, word_to_idx)
                    sim_sum += max(0, sim) 
                    valid_contexts += 1
            avg_sim = sim_sum / valid_contexts if valid_contexts > 0 else 0.0

        # --- PHẦN 2: ĐIỂM KENLM CỤC BỘ (QUAN TRỌNG NHẤT) ---
        # Chỉ lấy 2 từ trước và 2 từ sau của từ đang xét
        local_start = max(0, error_idx - 2)
        local_end = min(len(sentence), error_idx + 3)
        
        local_context = list(sentence[local_start:local_end])
        # Thay thế từ lỗi bằng candidate trong cửa sổ cục bộ
        local_context[error_idx - local_start] = candidate
        local_sentence_str = " ".join(local_context)
        
        # Tính điểm KenLM cho đoạn ngắn này
        # Vì đoạn này rất ngắn (3-5 từ), sự khác biệt giữa "đời sống" và "đời công" sẽ cực kỳ rõ rệt
        ken_score = kenlm_model.score(local_sentence_str)

        # --- PHẦN 3: ĐIỂM THƯỞNG TỪ GHÉP ---
        bigram_bonus = 0.0
        # Kiểm tra chặt chẽ hơn: chỉ thưởng nếu candidate tạo thành từ ghép VỚI TỪ SÁT CẠNH
        if error_idx > 0:
            if f"{sentence[error_idx-1].lower()}_{candidate.lower()}" in word_to_idx:
                bigram_bonus += 1.5 # Tăng mức thưởng để ưu tiên từ ghép đúng
        if error_idx < len(sentence) - 1:
            if f"{candidate.lower()}_{sentence[error_idx+1].lower()}" in word_to_idx:
                bigram_bonus += 1.5

        # --- PHẦN 4: TRỘN ĐIỂM ---
        # Tăng trọng số KenLM lên cao để nó làm "chủ cuộc chơi" cục bộ
        # Ken_score lúc này không chia cho len(sentence) nữa vì ta đã dùng cửa sổ hẹp
        total_score = (0.7 * ken_score) + (0.15 * avg_sim) + (0.15 * bigram_bonus)

        # Phạt từ quá ngắn (loại bỏ rác)
        if len(candidate) <= 1:
            total_score -= 5.0 

        if total_score > max_total_score:
            max_total_score = total_score
            best_word = candidate
        # In ra để debug (bạn sẽ thấy KenLM giờ đã có tiếng nói hơn)
        # print(f"Candidate: {candidate} | Total: {total_score:.4f} | Ken: {ken_score:.4f} | Sim: {avg_sim:.4f}  | Bonus: {bigram_bonus:.4f}")

    return best_word if best_word is not None else candidates[0]

Test hàm

In [75]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()
model = kenlm.Model(r"D:/NLP/project/3-gram-lm.binary")

candidates = lookup("hõc")
sentence = "hôm nay tôi đi hõc ở trường"
error_idx = 4
error_indices = [4]
sentence = sentence.split()

choose_best_candidate(candidates,
                      sentence,
                      error_idx,
                      embedding_matrix,
                      error_indices,
                      word_to_idx,
                      stopwords,
                      kenlm_model = model)

'học'

Tạo hàm để thực hiện thuật toán

In [76]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()
model_lm = kenlm.Model(r"D:/NLP/project/3-gram-lm.binary")

def correct_spelling_errors(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^\w\s_]', '', sentence)
    
    # Tìm vị trí các từ bị lỗi
    error = detect_error_word(sentence)

    sentence = sentence.split()

    # Bắt đầu quá trình sửa lỗi
    for error_idx in error:
        candidates = lookup(sentence[error_idx])

        # Gọi hàm và truyền đầy đủ các tham số cần thiết
        if candidates:
            sentence[error_idx] = choose_best_candidate(
                candidates=candidates,
                sentence=sentence,
                error_idx=error_idx,
                embedding_matrix=embedding_matrix,
                error_indices=error,
                word_to_idx=word_to_idx,
                stopwords=stopwords,
                kenlm_model = model_lm)

    # Dùng join để ghép chuỗi chuẩn xác và tối ưu hiệu năng
    sentence = " ".join(sentence)
    return sentence

Test hàm

In [128]:
# Test sửa lỗi
sentence = "Hôm nay tôi đi hõc ở trường."
fixed_sentence = correct_spelling_errors(sentence)

print(fixed_sentence)

hôm nay tôi đi học ở trường


In [129]:
# Test sửa lỗi
sentence = "sóng thền thì dất nguy hiểm"
fixed_sentence = correct_spelling_errors(sentence)

print(fixed_sentence)

sóng thần thì rất nguy hiểm


In [130]:
# Test sửa lỗi
sentence = "Nhiều giãi pháp đã áp dụng thành coong trong thực tiễn, có doanh thu tương dối lớn, phục vụ không chỏ cho doanh nghiệp mà còn cho đời sóng xã hội."
fixed_sentence = correct_spelling_errors(sentence)

print(fixed_sentence)

nhiều giải pháp đã áp dụng thành công trong thực tiễn có doanh thu tương đối lớn phục vụ không chỉ cho doanh nghiệp mà còn cho đời sống xã hội


# Kết hợp các thuật toán sửa lỗi
Biến các thuật toán thành 1 hàm sửa lỗi câu theo trường hợp
- 1. Sai lỗi chính tả  
- 2. Viết tắt
- 3. Viết không dấu     \
=> Tất cả sẽ dựa trên ngữ cảnh

# Test trên tập valid

Lấy top 1 câu sau chỉnh sửa để xem trên tập valid xem kết quả

# Chỉnh sửa phù hợp

Điều chỉnh thuật toán, dữ liệu, vv cho phù hợp lại

# Đánh giá lần cuối trên tập test